# Generative AI 010 — Runnables

Why does `prompt | model | parser` work at all? This notebook answers it three
ways: by counting, by rebuilding the idea from scratch, and by checking the real
library.

| Part | What we check |
|---|---|
| A | **42** ordered pairs and **343** three-step pipelines at 7 components |
| B | the problem in code — components that do not fit together |
| C | the whole idea rebuilt in ~40 lines of plain Python |
| D | chains nesting inside chains — the property that matters |
| E | the real library: `Runnable` at position **8** of a **12**-class ancestry |

Parts A–D need **no LangChain at all**. Part E needs `langchain-core`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from abc import ABC, abstractmethod

## Part A — Why the chain classes had to multiply

Before standardisation every component was called differently:

| Component | Called with |
|---|---|
| LLM | `.predict()` |
| PromptTemplate | `.format()` |
| Retriever | `.get_relevant_documents()` |
| OutputParser | `.parse()` |
| DocumentLoader | `.load()` |
| TextSplitter | `.split_documents()` |
| VectorStore | `.add_documents()` |

To join two of them you need glue that knows **both** interfaces. So you need
one glue class per ordered pair.

In [ ]:
print(f"{'components':>12}{'pairs to glue':>16}{'3-step pipelines':>20}")
for k in (2, 3, 4, 5, 6, 7):
    print(f"{k:>12}{k*(k-1):>16}{k**3:>20}")

n = 7
print(f"\nAt {n} components: {n*(n-1)} pairs, {n**3} three-step pipelines.")
print(f"With ONE shared interface: 1 connector, at any number of components.")
print(f"Reduction at {n} components: {n*(n-1)}x fewer glue classes.")

assert n*(n-1) == 42 and n**3 == 343

Nobody ships 343 classes. So you ship the popular ones — `LLMChain`,
`RetrievalQAChain`, `SimpleSequentialChain`, SQL chains, API chains — and the
library is permanently incomplete, with a learning curve to match.

> **What this is.** A counting argument about the *design*. LangChain 0.x is not
> installed here, so the actual number of chain classes it shipped is **not
> measured** — only the combinatorics that made the explosion inevitable.

## Part B — The problem, in code

In [ ]:
class FakeLLM:
    def predict(self, prompt):                      # LLMs used .predict()
        return {"response": "Delhi is the capital of India"}

class FakePromptTemplate:
    def __init__(self, template): self.template = template
    def format(self, d): return self.template.format(**d)   # templates used .format()

template = FakePromptTemplate("Write a {length} poem about {topic}")
llm = FakeLLM()

# By hand, every single time:
prompt = template.format({"length": "short", "topic": "India"})
print(llm.predict(prompt))

# So LangChain shipped a class to hide it:
class FakeLLMChain:
    def __init__(self, prompt, llm): self.prompt, self.llm = prompt, llm
    def run(self, d): return self.llm.predict(self.prompt.format(d))["response"]

print(FakeLLMChain(template, llm).run({"length": "short", "topic": "India"}))

It works, and it is **rigid**. `FakeLLMChain` holds exactly one prompt and one
LLM in that order. Want the LLM twice? A parser on the end? A retriever in
front? Each is a *different class*, because `.format()` and `.predict()` and
`.parse()` do not line up.

## Part C — The fix, from scratch

In [ ]:
class Runnable(ABC):
    """One shared interface. This is the entire idea."""
    @abstractmethod
    def invoke(self, input_data): ...
    def __or__(self, other):                    # this is what makes `a | b` work
        return RunnableSequence(self, other)

class RunnableSequence(Runnable):
    def __init__(self, *steps):
        self.steps = []
        for s in steps:                         # flatten so a|b|c is one sequence
            self.steps.extend(s.steps if isinstance(s, RunnableSequence) else [s])
    def invoke(self, x):
        for step in self.steps:
            x = step.invoke(x)                  # each output is the next input
        return x

class PromptT(Runnable):
    def __init__(self, t): self.t = t
    def invoke(self, d): return self.t.format(**d)

class LLM(Runnable):
    def __init__(self, replies): self.replies, self.i = replies, 0
    def invoke(self, p):
        r = self.replies[self.i % len(self.replies)]; self.i += 1
        return {"response": r}

class Parser(Runnable):
    def invoke(self, d): return d["response"]

chain = PromptT("Write a {length} poem about {topic}") | LLM(["a poem"]) | Parser()
print([type(s).__name__ for s in chain.steps])
print(chain.invoke({"length": "short", "topic": "India"}))

assert len(chain.steps) == 3

That is the whole mystery of the pipe operator: `__or__` returns a sequence,
and `invoke` walks it.

## Part D — Chains of chains

The fourth property, and the one that does the heavy lifting.

In [ ]:
joke    = PromptT("Write a joke about {topic}")
explain = PromptT("Explain this joke: {response}")
llm2    = LLM(["Why did the batsman cross the road?", "Because it was a googly."])
parser  = Parser()

chain1 = joke | llm2                   # topic -> {"response": a joke}
chain2 = explain | llm2 | parser       # joke  -> an explanation
final  = chain1 | chain2               # a chain OF chains

print("chain1 is a Runnable:", isinstance(chain1, Runnable))
print("final.invoke(...)   :", final.invoke({"topic": "cricket"}))
print("flattened steps     :", [type(s).__name__ for s in final.steps])

assert isinstance(chain1, Runnable)
assert len(final.steps) == 5

In [ ]:
# And what forces every component to comply:
class Broken(Runnable):
    pass                                # forgot to implement invoke

try:
    Broken()
except TypeError as e:
    print(type(e).__name__, "|", e)

# An abstract method is a GATE, not a comment. A component without invoke
# cannot be created - so it can never fail halfway through a pipeline.

**No new glue class was written for any of that.** One connector serves every
combination, at any length, including chains of chains. Compare with needing a
named class per pairing.

## Part E — Does the real library work this way?

In [ ]:
from langchain_core.runnables import (Runnable as LCRunnable, RunnableLambda,
                                      RunnableParallel)
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.language_models.fake_chat_models import FakeListChatModel

from langchain_anthropic import ChatAnthropic
mro = [c.__name__ for c in ChatAnthropic.__mro__]
print(len(mro), "classes in the MRO; Runnable at position", mro.index("Runnable"))
for i, name in enumerate(mro):
    print(f"   {i}. {name.split('[')[0]}" + ("   <-- here" if name == "Runnable" else ""))

assert "Runnable" in mro
# (BaseLanguageModel and RunnableSerializable appear twice: they are generic
#  classes, so the parameterised and plain forms are both in the MRO.)

In [ ]:
lc_chain = (PromptTemplate.from_template("{x}") | FakeListChatModel(responses=["a"])
            | StrOutputParser())

objects = {
    "PromptTemplate": PromptTemplate.from_template("{x}"),
    "ChatPromptTemplate": ChatPromptTemplate([("human", "{x}")]),
    "FakeListChatModel": FakeListChatModel(responses=["a"]),
    "StrOutputParser": StrOutputParser(),
    "JsonOutputParser": JsonOutputParser(),
    "RunnableLambda": RunnableLambda(lambda x: x),
    "RunnableSequence (a chain!)": lc_chain,
    "RunnableParallel": RunnableParallel({"a": lc_chain}),
}

print(f"{'object':<30}{'Runnable':>10}{'invoke':>8}{'batch':>7}{'stream':>8}")
for name, o in objects.items():
    print(f"{name:<30}{str(isinstance(o, LCRunnable)):>10}"
          f"{str(hasattr(o,'invoke')):>8}{str(hasattr(o,'batch')):>7}"
          f"{str(hasattr(o,'stream')):>8}")

assert all(isinstance(o, LCRunnable) for o in objects.values())

In [ ]:
pub = [m for m in dir(LCRunnable) if not m.startswith("_")]
print(len(pub), "public members on Runnable")
print("the load-bearing ones:",
      [m for m in ("invoke","ainvoke","batch","abatch","stream","astream",
                   "bind","with_retry","with_fallbacks","get_graph") if m in pub])

print()
print("invoke:", lc_chain.invoke({"x": "cricket"}))
print("batch :", lc_chain.batch([{"x": "a"}, {"x": "b"}, {"x": "c"}]))
print("stream:", list(lc_chain.stream({"x": "cricket"})))
print()
print("None of these were written for this chain. A component author")
print("implements ONE method and gets the rest of that surface free.")

## What to take away

- Chain classes multiplied because interfaces differed — **42** pairs and
  **343** three-step pipelines at 7 components, against **1** connector once the
  interface is shared.
- A **Runnable** has a purpose, a common interface, composability, and — the
  important one — **a workflow is itself a Runnable**.
- The pipe operator is just `__or__` returning a sequence, and a loop. ~40 lines
  rebuilds it.
- An **abstract method is a gate**: no `invoke`, no instance.
- The real library matches: `Runnable` at position **8** of 12, and all **8**
  component types carry `invoke`, `batch` and `stream`.

## Exercises

1. Extend the counting in Part A to 4-step pipelines. At what component count
   does the number pass a million?
2. Add a `RunnableParallel` to your from-scratch implementation. What does its
   `invoke` have to do differently?
3. Your `RunnableSequence` flattens nested sequences. Take the flattening out —
   does anything break, or is it purely cosmetic?
4. Add `batch` to the from-scratch `Runnable` base class so every subclass gets
   it free. How many lines? That is the payoff of a shared interface, measured.
5. `RunnableLambda` wraps a function. Implement it in your from-scratch version,
   then use it as the last step of a chain.